<a href="https://colab.research.google.com/github/VamsiKrishna-05/Crop-Recommendation-CCPS/blob/main/CCPS_Custom_ML_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# A validated custom DAG-SVM with FFO architecture
!pip install imbalanced-learn

# 📂 Imports
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE

# 📥 Load dataset
df = pd.read_csv("Crop Recommendation using Soil Properties and Weather Prediction.csv")

# 🧹 Encode categorical and target columns
df['Soilcolor'] = LabelEncoder().fit_transform(df['Soilcolor'])
df['label'] = LabelEncoder().fit_transform(df['label'])

X = df.drop(columns=['label'])
y = df['label']

# ⚖️ Standardize and balance data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

# 🍇 Optimized Fruit Fly Optimization (FFO)
def fast_ffo(X, y, generations=3, pop_size=5):
    best_score = 0
    best_C, best_gamma = 1, 0.1

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

    for gen in range(generations):
        print(f"🌀 Generation {gen+1}/{generations}")
        C_vals = np.random.uniform(0.1, 50, pop_size)
        gamma_vals = np.random.uniform(0.001, 2, pop_size)

        for i in range(pop_size):
            C, gamma = C_vals[i], gamma_vals[i]
            model = SVC(C=C, gamma=gamma, kernel='rbf')
            model.fit(X_train, y_train)
            score = model.score(X_val, y_val)
            if score > best_score:
                best_score = score
                best_C, best_gamma = C, gamma
                print(f"  ✅ New Best Score: {best_score:.4f} | C={C:.3f}, gamma={gamma:.3f}")
    return best_C, best_gamma

# 🔍 Find best C and gamma
best_C, best_gamma = fast_ffo(X_resampled, y_resampled)

# 🧠 DAG-based SVM Training
def train_dag_svm(X, y, C, gamma):
    classes = np.unique(y)
    models = {}
    for i in range(len(classes)):
        for j in range(i+1, len(classes)):
            class_i, class_j = classes[i], classes[j]
            mask = (y == class_i) | (y == class_j)
            X_pair, y_pair = X[mask], y[mask]
            y_binary = np.where(y_pair == class_i, 0, 1)
            clf = SVC(C=C, gamma=gamma, kernel='rbf')
            clf.fit(X_pair, y_binary)
            models[(class_i, class_j)] = clf
    return models, classes

# 🔮 DAG Prediction Logic
def predict_dag(models, x, class_list):
    remaining = list(class_list)
    while len(remaining) > 1:
        i, j = remaining[0], remaining[1]
        clf = models.get((i, j)) or models.get((j, i))
        x_input = x.reshape(1, -1)
        pred = clf.predict(x_input)[0]
        if (i, j) in models:
            if pred == 0:
                remaining.pop(1)
            else:
                remaining.pop(0)
        else:
            if pred == 0:
                remaining.pop(0)
            else:
                remaining.pop(1)
    return remaining[0]

# 🧪 Train DAG models
models, class_list = train_dag_svm(X_resampled, y_resampled, best_C, best_gamma)

# ✂️ Split test set
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# 🧠 Predict with DAG SVM
print("\n🔄 Predicting test samples...")
y_pred = [predict_dag(models, x, class_list) for x in X_test]

# 📊 Evaluate
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"\n✅ Final DAG-SVM Accuracy: {acc:.4f}")
print(f"🎯 Final DAG-SVM F1 Score: {f1:.4f}")


🌀 Generation 1/3
  ✅ New Best Score: 0.8360 | C=12.064, gamma=0.570
  ✅ New Best Score: 0.8549 | C=37.582, gamma=1.512
🌀 Generation 2/3
🌀 Generation 3/3

🔄 Predicting test samples...

✅ Final DAG-SVM Accuracy: 0.9964
🎯 Final DAG-SVM F1 Score: 0.9964


In [ ]:
# 📦 Install needed packages if running in Colab or Jupyter
# !pip install imbalanced-learn xgboost

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE

# 📂 Load dataset
df = pd.read_csv("Crop Recommendation using Soil Properties and Weather Prediction.csv")

# 🧹 Encode categorical variables
df['Soilcolor'] = LabelEncoder().fit_transform(df['Soilcolor'])
df['label'] = LabelEncoder().fit_transform(df['label'])

# 🎯 Feature and target split
X = df.drop(columns=['label'])
y = df['label']

# ⚖️ Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ➕ Apply SMOTE for class balancing
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

# ✂️ Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# 🧠 Initialize models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

# 📊 Train and evaluate
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results[name] = {"Accuracy": round(acc, 4), "F1 Score": round(f1, 4)}

# 📈 Show results
results_df = pd.DataFrame(results).T
print("🔍 Model Comparison with SMOTE + Scaling:\n")
print(results_df)


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [05:42:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


🔍 Model Comparison with SMOTE + Scaling:

                     Accuracy  F1 Score
Random Forest          0.8452    0.8398
XGBoost                0.8373    0.8328
Logistic Regression    0.4302    0.3942
